In [ ]:
PROJECT_ID = "gta-housing-508813"
DATASET = "raw_cmhc"

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

In [ ]:
import requests, pandas as pd, csv
from collections import Counter

OWNER = "Samriti5959"
REPO = "gta-housing-affordability"
BRANCH = "main"
FOLDER_PATH = "1-Get the data/raw/cmhc/files"

api_url = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{FOLDER_PATH}?ref={BRANCH}"
files = requests.get(api_url).json()

def build_column_names(header_cells):
    names, last_name, seen = [], "municipality", {}
    for i, cell in enumerate(header_cells):
        cell = cell.strip()
        if cell == "":
            base = "municipality" if i == 0 else f"{last_name}_flag"
        else:
            base = "".join(c if c.isalnum() else "_" for c in cell).strip("_")
            last_name = base
        if base in seen:
            seen[base] += 1
            base = f"{base}_{seen[base]}"
        else:
            seen[base] = 0
        names.append(base)
    return names

def clean_value(v):
    if v is None:
        return None
    v = v.strip().strip('"')
    if v in ("", "**", "*", "++", "n/a", "N/A"):
        return None
    v2 = v.replace(",", "")
    try:
        return float(v2)
    except ValueError:
        return v

def smart_read_csv(url):
    raw = requests.get(url).content
    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        text = raw.decode("latin1")

    rows = list(csv.reader(text.splitlines()))
    lengths = [len(r) for r in rows]
    non_trivial = [l for l in lengths if l > 1]
    if not non_trivial:
        return pd.DataFrame()

    target_width = Counter(non_trivial).most_common(1)[0][0]
    header_idx = next(i for i, l in enumerate(lengths) if l == target_width)
    col_names = build_column_names(rows[header_idx])

    data_rows = []
    for row in rows[header_idx + 1:]:
        if not row or not any(c.strip() for c in row):
            continue
        if row[0].strip().lower() in ("notes", "source"):
            break
        row = (row + [None] * len(col_names))[:len(col_names)]
        data_rows.append([clean_value(v) for v in row])

    return pd.DataFrame(data_rows, columns=col_names)

for f in files:
    if f["name"].endswith(".csv"):
        table_name = f["name"].replace(".csv", "").lower()
        try:
            df = smart_read_csv(f["download_url"])
            table_ref = f"{PROJECT_ID}.{DATASET}.{table_name}"
            client.load_table_from_dataframe(df, table_ref).result()
            print(f"Loaded {f['name']} -> {table_ref}")
        except Exception as e:
            print(f"FAILED on {f['name']}: {e}")

Loaded rent_by_csd_toronto_cma_2025.csv -> gta-housing-508813.raw_cmhc.rent_by_csd_toronto_cma_2025
Loaded rent_by_zone_toronto_cma_2025.csv -> gta-housing-508813.raw_cmhc.rent_by_zone_toronto_cma_2025
Loaded rent_history_3518001_pickering.csv -> gta-housing-508813.raw_cmhc.rent_history_3518001_pickering
Loaded rent_history_3518005_ajax.csv -> gta-housing-508813.raw_cmhc.rent_history_3518005_ajax
Loaded rent_history_3518009_whitby.csv -> gta-housing-508813.raw_cmhc.rent_history_3518009_whitby
Loaded rent_history_3518013_oshawa.csv -> gta-housing-508813.raw_cmhc.rent_history_3518013_oshawa
Loaded rent_history_3518017_clarington.csv -> gta-housing-508813.raw_cmhc.rent_history_3518017_clarington
Loaded rent_history_3518020_scugog.csv -> gta-housing-508813.raw_cmhc.rent_history_3518020_scugog
Loaded rent_history_3518029_uxbridge.csv -> gta-housing-508813.raw_cmhc.rent_history_3518029_uxbridge
Loaded rent_history_3518039_brock.csv -> gta-housing-508813.raw_cmhc.rent_history_3518039_brock
Lo

In [ ]:
import re

def sanitize_columns(df):
    seen = {}
    new_cols = []
    for c in df.columns:
        c2 = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in str(c).strip()).strip("_")
        if c2 == "" or c2[0].isdigit():
            c2 = f"col_{c2}"
        if c2 in seen:
            seen[c2] += 1
            c2 = f"{c2}_{seen[c2]}"
        else:
            seen[c2] = 0
        new_cols.append(c2)
    df.columns = new_cols
    return df

def read_plain_csv(url):
    try:
        df = pd.read_csv(url, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(url, encoding="latin1")
    return sanitize_columns(df)

def build_meta_column_names(header_cells):
    names, seen = [], {}
    for cell in header_cells:
        base = "".join(c if c.isalnum() or c == "_" else "_" for c in cell.strip()).strip("_")
        if base == "":
            base = "col"
        if base[0].isdigit():
            base = f"col_{base}"
        if base in seen:
            seen[base] += 1
            base = f"{base}_{seen[base]}"
        else:
            seen[base] = 0
        names.append(base)
    return names

def clean_meta_value(v):
    if v is None:
        return None
    v = v.strip()
    return v if v != "" else None

def split_into_sections(rows):
    sections, current = [], []
    for row in rows:
        if not row or not any(c.strip() for c in row):
            if current:
                sections.append(current)
                current = []
            continue
        current.append(row)
    if current:
        sections.append(current)
    return sections

def load_metadata_csv(url, base_table_name):
    raw = requests.get(url).content
    try:
        text = raw.decode("utf-8-sig")
    except UnicodeDecodeError:
        text = raw.decode("latin1")

    rows = list(csv.reader(text.splitlines()))
    sections = split_into_sections(rows)

    loaded = []
    for i, section in enumerate(sections, start=1):
        header, data_rows = section[0], section[1:]
        if not data_rows:
            continue
        col_names = build_meta_column_names(header)
        clean_rows = []
        for row in data_rows:
            row = (row + [None] * len(col_names))[:len(col_names)]
            clean_rows.append([clean_meta_value(v) for v in row])
        df = pd.DataFrame(clean_rows, columns=col_names)
        table_name = f"{base_table_name}_meta_{i}"
        table_ref = f"{PROJECT_ID}.{DATASET}.{table_name}"
        client.load_table_from_dataframe(df, table_ref).result()
        loaded.append(table_name)
    return loaded

STATCAN_FOLDER = "1-Get the data/raw/statcan"
api_url2 = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{STATCAN_FOLDER}?ref={BRANCH}"
items = requests.get(api_url2).json()

for item in items:
    name = item.get("name", "")
    if item.get("type") != "file" or not name.endswith(".csv"):
        continue
    base_name = name.replace(".csv", "").lower()
    try:
        if name.lower().endswith("_metadata.csv"):
            tables = load_metadata_csv(item["download_url"], base_name)
            print(f"Loaded {name} -> tables: {tables}")
        else:
            df = read_plain_csv(item["download_url"])
            table_ref = f"{PROJECT_ID}.{DATASET}.{base_name}"
            client.load_table_from_dataframe(df, table_ref).result()
            print(f"Loaded {name} -> table '{base_name}'")
    except Exception as e:
        print(f"FAILED on {name}: {e}")

Loaded 11100009_MetaData.csv -> tables: ['11100009_metadata_meta_1', '11100009_metadata_meta_2', '11100009_metadata_meta_3', '11100009_metadata_meta_4', '11100009_metadata_meta_5', '11100009_metadata_meta_6', '11100009_metadata_meta_7', '11100009_metadata_meta_8']
Loaded 11100009_gta.csv -> table '11100009_gta'
Loaded 17100155_MetaData.csv -> tables: ['17100155_metadata_meta_1', '17100155_metadata_meta_2', '17100155_metadata_meta_3', '17100155_metadata_meta_4', '17100155_metadata_meta_5', '17100155_metadata_meta_6', '17100155_metadata_meta_7']
Loaded 17100155_gta.csv -> table '17100155_gta'
Loaded 18100004_MetaData.csv -> tables: ['18100004_metadata_meta_1', '18100004_metadata_meta_2', '18100004_metadata_meta_3', '18100004_metadata_meta_4', '18100004_metadata_meta_5', '18100004_metadata_meta_6', '18100004_metadata_meta_7']
Loaded 18100004_gta.csv -> table '18100004_gta'
Loaded 34100133_MetaData.csv -> tables: ['34100133_metadata_meta_1', '34100133_metadata_meta_2', '34100133_metadata_m